# 11.33 Decision transformers

Decision transformers recast control as predicting the next action in a trajectory.

Reinforcement learning is where a prediction changes what data arrives next. Probability supplies transition probabilities and expectations; optimization supplies iterative improvement. These notebooks keep the environments tiny and CPU-only while implementing the real decision rule. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1135
random.seed(SEED)
np.random.seed(SEED)

ACTIONS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
ACTION_NAMES = ["up", "right", "down", "left"]


@dataclass
class GridEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    hazards: set
    traps: set
    slip: float
    wind: float
    budget: float
    max_steps: int
    reward_goal: float = 5.0
    reward_step: float = -0.05
    cost_hazard: float = 1.0

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return 4

    def state_to_pos(self, state):
        row = state // self.width
        col = state % self.width
        return (row, col)

    def pos_to_state(self, pos):
        return pos[0] * self.width + pos[1]

    def in_bounds(self, pos):
        row, col = pos
        return 0 <= row < self.height and 0 <= col < self.width

    def start_state(self):
        return self.pos_to_state(self.start)

    def goal_state(self):
        return self.pos_to_state(self.goal)

    def transition(self, state, action, slip_override=None):
        slip = self.slip if slip_override is None else slip_override
        probs = []
        primary = self._move(state, action)
        probs.append((1.0 - slip, primary))
        side_actions = [(action + 1) % 4, (action - 1) % 4]
        for side in side_actions:
            probs.append((slip / 2.0, self._move(state, side)))
        merged = {}
        for prob, next_state in probs:
            merged[next_state] = merged.get(next_state, 0.0) + prob
        return list(merged.items())

    def _move(self, state, action):
        if state == self.goal_state():
            return state
        pos = np.array(self.state_to_pos(state))
        next_pos = tuple(pos + ACTIONS[action])
        if self.wind > 0 and action == 1:
            next_pos = (max(0, next_pos[0] - 1), next_pos[1])
        if not self.in_bounds(next_pos):
            next_pos = tuple(pos)
        return self.pos_to_state(next_pos)

    def reward_cost_done(self, state, action, next_state):
        pos = self.state_to_pos(next_state)
        done = next_state == self.goal_state()
        reward = self.reward_goal if done else self.reward_step
        cost = self.cost_hazard if pos in self.hazards or pos in self.traps else 0.0
        if pos in self.traps:
            reward -= 1.0
        return reward, cost, done


def make_f12_ladder():
    return [
        GridEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), {(0, 1)}, set(), 0.00, 0.0, 1.0, 3),
        GridEnv("D2 slippery 3-state hazards", 1, 3, (0, 0), (0, 2), {(0, 1)}, set(), 0.10, 0.0, 1.0, 6),
        GridEnv("D3 4x4 gridworld hazards", 4, 4, (3, 0), (0, 3), {(2, 1), (1, 2)}, set(), 0.08, 0.0, 1.2, 18),
        GridEnv("D4 stochastic windy grid", 5, 5, (4, 0), (0, 4), {(3, 1), (2, 2), (1, 3)}, set(), 0.15, 0.25, 1.4, 28),
        GridEnv("D5 sparse grid with traps", 6, 6, (5, 0), (0, 5), {(4, 1), (3, 2), (2, 3)}, {(1, 4), (4, 4)}, 0.18, 0.30, 1.5, 40),
    ]


def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    for step, reward in enumerate(rewards):
        total += (gamma ** step) * reward
    return total


def value_iteration(env, gamma=0.9, penalty=0.0, slip_override=None, iterations=80):
    values = np.zeros(env.n_states)
    q_values = np.zeros((env.n_states, env.n_actions))
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            if state == env.goal_state():
                continue
            action_scores = []
            for action in range(env.n_actions):
                score = 0.0
                for prob, next_state in env.transition(state, action, slip_override):
                    reward, cost, done = env.reward_cost_done(state, action, next_state)
                    bootstrap = 0.0 if done else gamma * values[next_state]
                    score += prob * (reward - penalty * cost + bootstrap)
                action_scores.append(score)
            new_values[state] = max(action_scores)
            q_values[state] = action_scores
        values = new_values
    policy = np.argmax(q_values, axis=1)
    return values, q_values, policy


def evaluate_policy(env, policy, gamma=0.9, episodes=40, seed=0, slip_override=None):
    rng = np.random.default_rng(seed)
    returns = []
    costs = []
    wins = []
    paths = []
    for episode in range(episodes):
        state = env.start_state()
        rewards = []
        cost_values = []
        path = [state]
        for step in range(env.max_steps):
            action = int(policy[state])
            transitions = env.transition(state, action, slip_override)
            probs = np.array([item[0] for item in transitions])
            idx = rng.choice(len(transitions), p=probs)
            next_state = transitions[idx][1]
            reward, cost, done = env.reward_cost_done(state, action, next_state)
            rewards.append(reward)
            cost_values.append(cost)
            state = next_state
            path.append(state)
            if done:
                break
        returns.append(discounted_return(rewards, gamma))
        costs.append(sum(cost_values))
        wins.append(float(state == env.goal_state()))
        paths.append(path)
    return {
        "return": float(np.mean(returns)),
        "cost": float(np.mean(costs)),
        "win_rate": float(np.mean(wins)),
        "path": paths[0],
    }



def preview_ladder(ladder):
    rows = []
    for idx, env in enumerate(ladder, start=1):
        rows.append({
            "rung": f"D{idx}",
            "name": env.name,
            "shape": (env.height, env.width),
            "states": env.n_states,
            "slip": env.slip,
            "budget": env.budget,
            "hazards": len(env.hazards) + len(env.traps),
        })
    return rows


def plot_env_panel(ax, env, values, title):
    image = np.asarray(values).reshape(env.height, env.width)
    ax.imshow(image, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for hazard in env.hazards:
        ax.text(hazard[1], hazard[0], "H", color="white", ha="center", va="center")
    for trap in env.traps:
        ax.text(trap[1], trap[0], "T", color="red", ha="center", va="center")
    ax.text(env.start[1], env.start[0], "S", color="white", ha="center", va="center")
    ax.text(env.goal[1], env.goal[0], "G", color="white", ha="center", va="center")


## The concept, built once (D1)

The lesson object is $$p(a_t\mid R_t,s_1,a_1,\ldots,s_t)$$. We assert the exact RL arithmetic, then fit a tiny return-conditioned sequence classifier.

In [ ]:

rewards = [1, 0, 2]
gamma = 0.9
G = discounted_return(rewards, gamma)
y = 1 + gamma * 0.8
q_new = 0.4 + 0.5 * (y - 0.4)
probs = softmax([1, 0])
expected_reward = probs[0] * 2 + probs[1] * 0
ucb = 0.55 + math.sqrt(2 * math.log(20) / 5)
print("G", round(G, 3))
print("target", round(y, 3))
print("Q_new", round(q_new, 3))
print("policy", np.round(probs, 3))
print("expected reward", round(float(expected_reward), 3))
print("UCB", round(ucb, 3))
assert round(G, 3) == 2.620
assert round(y, 3) == 1.720
assert round(q_new, 3) == 1.060
assert round(float(probs[0]), 3) == 0.731
assert round(float(probs[1]), 3) == 0.269
assert round(float(expected_reward), 3) == 1.462
assert round(ucb, 3) == 1.645


On D1, the model conditions action prediction on return-to-go plus trajectory context instead of running online exploration.

In [ ]:
def generate_trajectory_dataset(env, count=120, gamma=0.9, seed=0):
    rng = np.random.default_rng(seed)
    expert_values, expert_q, expert_policy = value_iteration(env, gamma=gamma, iterations=90)
    rows = []
    returns = []
    for episode in range(count):
        state = env.start_state()
        states = []
        actions = []
        rewards = []
        previous_action = 0
        for step in range(env.max_steps):
            if rng.random() < 0.72:
                action = int(expert_policy[state])
            else:
                action = int(rng.integers(env.n_actions))
            transitions = env.transition(state, action)
            probs = np.array([item[0] for item in transitions])
            idx = rng.choice(len(transitions), p=probs)
            next_state = transitions[idx][1]
            reward, cost, done = env.reward_cost_done(state, action, next_state)
            states.append(state)
            actions.append(action)
            rewards.append(reward)
            state = next_state
            previous_action = action
            if done:
                break
        rtg = []
        for index in range(len(rewards)):
            rtg.append(discounted_return(rewards[index:], gamma))
        for index, state_value in enumerate(states):
            rows.append((state_value, actions[index], rtg[index], index, previous_action))
        returns.append(discounted_return(rewards, gamma))
    return rows, np.array(returns)


def featurize_dt(state, rtg, step, previous_action, env):
    state_one_hot = np.zeros(env.n_states)
    state_one_hot[state] = 1.0
    action_one_hot = np.zeros(env.n_actions)
    action_one_hot[previous_action] = 1.0
    scaled = np.array([rtg / 6.0, step / max(1, env.max_steps)])
    return np.concatenate([state_one_hot, action_one_hot, scaled])


def return_conditioned_policy(env, requested_return=4.0, epochs=90, lr=0.15, seed=0, clip_to_support=True):
    rows, returns = generate_trajectory_dataset(env, count=140, seed=seed)
    min_return = float(np.quantile(returns, 0.05))
    max_return = float(np.quantile(returns, 0.95))
    target_return = float(requested_return)
    if clip_to_support:
        target_return = float(np.clip(target_return, min_return, max_return))
    feature_dim = env.n_states + env.n_actions + 2
    weights = np.zeros((feature_dim, env.n_actions))
    for epoch in range(epochs):
        for state, action, rtg, step, previous_action in rows:
            features = featurize_dt(state, rtg, step, previous_action, env)
            probs = softmax(features @ weights)
            grad = probs
            grad[action] -= 1.0
            weights -= lr * np.outer(features, grad) / len(rows)
    policy = np.zeros(env.n_states, dtype=int)
    for state in range(env.n_states):
        features = featurize_dt(state, target_return, 0, 0, env)
        policy[state] = int(np.argmax(features @ weights))
    support = {"min": min_return, "max": max_return, "target": target_return}
    return policy, weights, support, returns

env = make_f12_ladder()[0]
policy, weights, support, returns = return_conditioned_policy(env, requested_return=4.0, seed=SEED)
metrics = evaluate_policy(env, policy, seed=SEED)
print("support", support)
print("metrics", metrics)
assert weights.shape[1] == env.n_actions

## The dataset ladder

Family F12 uses a D1–D5 sequential-decision ladder inline: a two-state chain, slippery chain, 4x4 grid, windy grid, and sparse trap grid.

In [ ]:
ladder = make_f12_ladder()
for row in preview_ladder(ladder):
    print(row)
print("sample D5 policy grid shape", (ladder[-1].height, ladder[-1].width))

## Run the SAME method across D1-D5

Apply the same method to every rung and collect the plan metric.

In [ ]:
results = []
artifacts = []
for index, env in enumerate(ladder, start=1):
    requested = 4.0 + 0.2 * index
    policy, weights, support, returns = return_conditioned_policy(env, requested_return=requested, seed=SEED + index)
    metrics = evaluate_policy(env, policy, seed=SEED + index)
    row = {
        "rung": f"D{index}",
        "requested": requested,
        "target": support["target"],
        "return": metrics["return"],
        "win_rate": metrics["win_rate"],
    }
    results.append(row)
    artifacts.append((env, policy, returns, support))
print("rung | requested | clipped_target | achieved_return | win_rate")
for row in results:
    print(row["rung"], round(row["requested"], 3), round(row["target"], 3), round(row["return"], 3), round(row["win_rate"], 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (env, policy, returns, support) in enumerate(artifacts):
    policy_values = policy.reshape(env.height, env.width)
    plot_env_panel(axes[0, col], env, policy_values, f"D{col + 1} action")
    axes[1, col].hist(returns, bins=10, color="gray")
    axes[1, col].axvline(support["target"], color="red")
    axes[1, col].set_title(f"D{col + 1} return support")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
xs = np.arange(1, 6)
plt.plot(xs, [row["requested"] for row in results], marker="o", label="requested")
plt.plot(xs, [row["return"] for row in results], marker="s", label="achieved")
plt.xticks(xs, [row["rung"] for row in results])
plt.xlabel("rung")
plt.ylabel("return")
plt.title("Achieved return vs requested return")
plt.legend()
plt.show()

## Pitfall on D5: ignoring policy support

Requesting a return absent from the offline logs asks the model to extrapolate. The fix clips the return condition to observed support and reports the clipped target.

In [ ]:
env = ladder[-1]
unsupported_policy, unsupported_weights, unsupported_support, unsupported_returns = return_conditioned_policy(env, requested_return=20.0, seed=123, clip_to_support=False)
unsupported = evaluate_policy(env, unsupported_policy, seed=123)
clipped_policy, clipped_weights, clipped_support, clipped_returns = return_conditioned_policy(env, requested_return=20.0, seed=123, clip_to_support=True)
fixed = evaluate_policy(env, clipped_policy, seed=123)
print("unsupported support", unsupported_support)
print("unsupported", unsupported)
print("clipped support", clipped_support)
print("fixed", fixed)

## Evaluate it + Practice

- Compare the reported return / support gap / win-rate against a no-skill baseline such as a random or immediate-reward policy.
- Sanity check that the D1 result matches the exact lesson arithmetic before trusting harder rungs.
- Ablate the key idea, such as removing constraints, randomization, return conditioning, population replay, or search.
- Watch failure signals: budget violations, transfer collapse, unsupported target returns, non-stationary opponents, or shallow reward chasing.

Practice:
1. Change the discount from 0.9 to 0.8 and predict which rung changes most.


2. Add one hazard or trap to D4 and rerun only the small table, not a long training job.

3. Replace the no-skill baseline with a hand-written safe policy and compare the metric.